# BioGPT Discharge Summary -- Colab Runner

Runs the full pipeline end to end on a Colab T4 GPU: data prep -> zero-shot baseline demo -> LoRA fine-tuning -> generation -> evaluation.

**Before you start:**
- If you have credentialed access to **MIMIC-IV-Ext-BHC** on PhysioNet, download it yourself (this notebook never does so automatically -- that would violate the Data Use Agreement) and upload the CSV to `data/raw/` in a later cell, then set `dataset.mode: mimic_bhc` in `config.yaml`.
- Otherwise, leave `config.yaml` on `dataset.mode: mtsamples`, the public fallback -- no credentialed access needed.
- Runtime > Change runtime type > GPU (T4) before running.

Each step below is also an ordinary script in `src/` -- this notebook is just an orchestrator, so you can always re-run an individual step from a terminal instead.

In [ ]:
!nvidia-smi

## Get the repo onto this runtime

Two options -- set `REPO_URL` below if your code is on GitHub, or leave it blank to upload a zip of the local project folder instead.

In [ ]:
import os

REPO_URL = ""  # e.g. "https://github.com/<you>/biogpt-discharge-summary.git" -- leave blank to upload instead
REPO_DIR = "/content/biogpt-discharge-summary"

if REPO_URL:
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
else:
    if not os.path.isdir(REPO_DIR):
        from google.colab import files
        print("Upload a zip of the biogpt-discharge-summary project folder (e.g. right-click the folder locally, ")
        print("Compress / Zip, then upload the .zip here).")
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        import zipfile
        with zipfile.ZipFile(zip_name) as zf:
            zf.extractall("/content/")
        # Handles the common case where the zip contains a single top-level folder.
        extracted = [d for d in os.listdir("/content") if "biogpt-discharge-summary" in d and os.path.isdir(f"/content/{d}")]
        if extracted and extracted[0] != "biogpt-discharge-summary":
            os.rename(f"/content/{extracted[0]}", REPO_DIR)

%cd {REPO_DIR}
!pip install -q -r requirements.txt

## Mount Google Drive for checkpoint persistence

Colab's free-tier T4 sessions can disconnect or time out, and `checkpoints/` is **not** committed to git (see `.gitignore`). Symlinking `checkpoints/` to a Drive folder means a dropped session doesn't cost you a from-scratch retrain -- `train.py` keeps writing to the same relative `checkpoints/` path, which now transparently points at Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CKPT_DIR = "/content/drive/MyDrive/biogpt-discharge-summary-checkpoints"
!mkdir -p {DRIVE_CKPT_DIR}
!rm -rf {REPO_DIR}/checkpoints
!ln -sfn {DRIVE_CKPT_DIR} {REPO_DIR}/checkpoints
print("checkpoints/ now points at Drive:", os.path.realpath(f"{REPO_DIR}/checkpoints"))

## (Optional) Upload MIMIC-IV-Ext-BHC

Skip this cell entirely if you're using the `mtsamples` fallback (the default in `config.yaml`). If you have your own PhysioNet-downloaded MIMIC-IV-Ext-BHC CSV, upload it here and make sure `dataset.mode: mimic_bhc` is set in `config.yaml` (and that `dataset.mimic_bhc.input_column` / `target_column` match your export's actual column names).

In [ ]:
UPLOAD_MIMIC = False  # flip to True to run this cell's upload

if UPLOAD_MIMIC:
    from google.colab import files
    os.makedirs("data/raw", exist_ok=True)
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, f"data/raw/{fname}")
    print("Uploaded to data/raw/. Update config.yaml's dataset.mimic_bhc.raw_path if the filename differs.")

## Step 1: Data prep

Loads whichever dataset `config.yaml` points at, cleans text, builds (input, target) pairs, splits 80/10/10 with a fixed seed, and reports token-length stats (using BioGPT's own tokenizer) including how many examples get truncated against the 768/256 token limits. See `src/data_prep.py` for the full reasoning, especially around why the MTSamples section-split is a heuristic that *drops* rows it can't confidently parse rather than guessing.

In [ ]:
!python src/data_prep.py

In [ ]:
import json
from IPython.display import Image, display

print(json.dumps(json.load(open("outputs/dataset_stats.json")), indent=2))
display(Image("outputs/figures/token_length_histogram.png"))

## Step 2: Zero-shot baseline demo

Runs the base (not fine-tuned) BioGPT on three synthetic clinical prompts with greedy vs. sampling decoding, to make concrete the determinism-vs-variance point before any fine-tuning happens. No dataset or GPU training required for this step -- it's also a good smoke test that the environment and base model load correctly.

In [ ]:
!python src/baseline_demo.py

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("outputs/baseline_demo.md").read()))

## Step 3: LoRA fine-tuning

Fine-tunes BioGPT with LoRA adapters on the attention projections (`q_proj`, `k_proj`, `v_proj`, `out_proj` -- confirmed against BioGPT's source, printed again at the start of this run as a runtime sanity check). The loss is masked to the target (Brief Hospital Course) tokens only, so the model learns to *summarize*, not to reproduce the input note. Expect this to take a while on a T4 -- `config.yaml`'s `training` section controls batch size / epochs / etc. if you need to shorten it for a quick test.

In [ ]:
!python src/train.py

In [ ]:
print(json.dumps(json.load(open("outputs/train_summary.json")), indent=2))
display(Image("outputs/figures/loss_curve.png"))

## Step 4: Generation on the test set

Generates summaries for every test example with the base model (zero-shot, greedy) and the fine-tuned model (greedy and sampling), reusing a single loaded model via `disable_adapter()` to keep GPU memory down. If this is taking too long on the free tier, lower `generation.max_test_examples` in `config.yaml` to a smaller number and re-run.

In [ ]:
!python src/generate.py

In [ ]:
import pandas as pd
pd.read_csv("outputs/test_generations.csv").head(3)

## Step 5: Evaluation

ROUGE-1/2/L, BERTScore, the approximate hallucination proxy (explicitly *not* a validated factuality metric -- see `src/evaluate.py`'s docstring), and a sampling-reproducibility check (mean pairwise ROUGE-L between repeated sampling runs on the same prompt vs. repeated greedy runs). This is the step that produces the headline numbers for the model card.

In [ ]:
!python src/evaluate.py

In [ ]:
print(json.dumps(json.load(open("outputs/metrics.json")), indent=2))
display(Image("outputs/figures/metrics_comparison.png"))

In [ ]:
display(Markdown(open("outputs/qualitative_examples.md").read()))

## What you now have for the Model Card

All under `outputs/` (and persisted checkpoints under Drive via the symlink set up above):

- `dataset_stats.json`, `figures/token_length_histogram.png` -- Data section
- `baseline_demo.md` -- "Working Principle in Action" (greedy determinism vs. sampling variance)
- `train_summary.json` (incl. trainable/total param counts), `figures/loss_curve.png`, `trainer_state.json` -- Training Details
- `test_generations.csv` -- raw generations, all four columns, for spot-checking
- `metrics.json`, `figures/metrics_comparison.png` -- Quantitative Metrics + Reproducibility
- `qualitative_examples.md` -- Qualitative Analysis / Limitations (good + bad examples, hallucination-proxy flags)

Every number in these files came from an actual run of this notebook -- none of it is fabricated or hand-written, which is exactly what you want to be able to say in the model card's evaluation section.

## Step 6: Download results + trained model

Colab deletes everything outside Drive when the session ends. This zips `outputs/` (metrics, figures, generations) **and** the trained LoRA adapter (`checkpoints/lora_adapter/`, a few MB) into one file, copies it to Drive, and downloads it.

On your computer, unzip it into the project folder (`biogpt-discharge-summary/`) so you get `outputs/` and `checkpoints/lora_adapter/` -- the local app (`start_app.bat`) loads the adapter from there.

In [ ]:
!rm -f /content/biogpt_results.zip
!zip -r -q /content/biogpt_results.zip outputs checkpoints/lora_adapter
!cp /content/biogpt_results.zip "{DRIVE_CKPT_DIR}/"
!ls -lh /content/biogpt_results.zip

from google.colab import files
files.download("/content/biogpt_results.zip")